# SecOps: Agentic Automation & Security Playbooks (March 2026 Preview)

[![Open In Colab](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/secops_agentic_automation_demo.ipynb)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/secops_agentic_automation_demo.ipynb)

This notebook demonstrates **Agentic Automation** for Security Operations (SecOps), integrating AI Agents into deterministic playbooks to handle threat detection and response. It also showcases the **ADK 1.28 Slack integration** for ChatOps-style incident alerting.

## Use Case
A security analyst needs to investigate a potential brute-force attack. Instead of a purely manual or purely automated response, they use an **Agentic Playbook** that:
1.  **Detects**: Triggers based on a high number of failed login attempts.
2.  **Investigates (AI Agent)**: An agent analyzes the IP reputation and cross-references logs to determine the severity.
3.  **Responds (Deterministic)**: If the agent confirms a high risk, the playbook automatically blocks the IP.
4.  **Informs**: Sends an incident summary (via Slack if configured, otherwise logged to console).

### Release Notes
- [Google SecOps — Agentic Automation](https://docs.cloud.google.com/chronicle/docs/soar/respond/working-with-playbooks/agentic-automation) — AI agents embedded in Chronicle SOAR playbooks for automated threat investigation and response
- [ADK v1.28.0](https://github.com/google/adk-python/releases/tag/v1.28.0) — Native Slack integration; used as the agent framework for this demo

### Requirements
- `google-adk >= 1.28.0` installed.
- Gemini 3.1 Pro (Preview) access.
- **Optional**: Slack Bot Token (`SLACK_BOT_TOKEN`) and channel ID for live notifications. The demo runs fully without Slack — notifications are printed to console by default.

In [ ]:
# 1. Setup and Authentication
%pip install "google-adk>=1.28.0" google-genai nest-asyncio --quiet --index-url https://pypi.org/simple

try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab')
except ModuleNotFoundError:
    print('Not running in Colab — using Application Default Credentials (ADC)')

import os
import nest_asyncio
nest_asyncio.apply()

project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
location = 'us-central1'  # @param {type:"string"}
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = location
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

### 2. [PREREQUISITES] Initialize SecOps Agent with Optional Slack Notification

We initialize a SecOps agent for log analysis. If `SLACK_BOT_TOKEN` is set, the notification step will post to Slack; otherwise it logs to console.

In [ ]:
from google.adk import Agent, Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai import types
import os

# --- Optional Slack Setup (ADK 1.28 Feature) ---
SLACK_BOT_TOKEN = os.environ.get("SLACK_BOT_TOKEN")  # Set to enable live Slack alerts
SLACK_CHANNEL = os.environ.get("SLACK_CHANNEL", "#secops-alerts")

def notify_incident(summary: str):
    """Send incident notification — Slack if configured, console otherwise."""
    if SLACK_BOT_TOKEN:
        try:
            import urllib.request, json
            req = urllib.request.Request(
                "https://slack.com/api/chat.postMessage",
                data=json.dumps({"channel": SLACK_CHANNEL, "text": f":rotating_light: *SecOps Alert*\n{summary}"}).encode(),
                headers={"Authorization": f"Bearer {SLACK_BOT_TOKEN}", "Content-Type": "application/json"}
            )
            urllib.request.urlopen(req)
            print(f"[Slack] Alert sent to {SLACK_CHANNEL}")
        except Exception as e:
            print(f"[Slack] Failed to send ({e}), falling back to console.")
            print(f"[Console] {summary}")
    else:
        print(f"[Console] {summary}")
        print("  (Set SLACK_BOT_TOKEN to enable live Slack notifications)")

# 1. Define the SecOps Analyst Agent
secops_agent = Agent(
    model="gemini-3.1-pro-preview",
    name="SecOpsAnalyst",
    instruction="""
    You are a specialized security analyst.
    Your goal is to investigate alerts and determine if they represent a real threat.
    Assess the severity based on IP reputation, login patterns, and geographic anomalies.
    """
)

# 2. Initialize Runner (Industrialized Standard)
runner = Runner(
    agent=secops_agent,
    session_service=InMemorySessionService(),
    app_name="secops_automation_demo",
    auto_create_session=True
)

async def run_secops_playbook():
    print("--- Executing Agentic Playbook: Brute Force Investigation ---")
    
    # Step 1: Deterministic Trigger & Log Fetching
    print("[Step: Trigger] High volume of failed logins detected for user 'admin'.")
    log_data = "Failed login from IP 192.168.1.100 (Location: Unknown) at 10:00, 10:01, 10:02..."
    
    # Step 2: AI Investigation (via Runner)
    print("\n[Step: AI Investigation] Analyzing logs and IP reputation...")
    prompt = f"Investigate these logs for threat severity: {log_data}"
    
    message = types.Content(parts=[types.Part(text=prompt)], role='user')
    async for event in runner.run_async(
        user_id="partner_user",
        session_id="march_session",
        new_message=message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"Agent: {part.text}")

    # Step 3: Deterministic Remediation
    print("\n[Step: Auto-Remediation] ACTION: IP 192.168.1.100 has been blocked in Cloud Armor.")
    
    # Step 4: Notification (Slack if configured, console otherwise)
    notify_incident("Critical Brute Force attack from 192.168.1.100 — IP blocked in Cloud Armor.")

await run_secops_playbook()

### 4. Things to remember or know
- **Hybrid workflows**: Combine AI reasoning with deterministic code — the agent investigates, but critical actions (like blocking IPs) only run when specific conditions are met.
- **Slack integration (ADK 1.28)**: Agents can post alerts directly to Slack channels. This demo supports it opt-in — set `SLACK_BOT_TOKEN` to enable, or run without it for console output.
- **Runner pattern**: All March 2026 demos use the `Runner` for automatic session management and event streaming.
- **Availability**: Public Preview as of March 20, 2026.